# FTS-Diffusion: Financial Time Series Generation with Diffusion Models
## Google Colab Training & Evaluation Notebook

This notebook implements the complete FTS-Diffusion pipeline:
1. **Pattern Recognition** (SISC clustering)
2. **Generation Module** (Scaling Autoencoder + Conditional Diffusion)
3. **Evolution Module** (Pattern Markov transition network)
4. **Sampling** (Synthetic time series generation)
5. **Evaluation** (Stylized facts analysis)


## Setup & Environment

In [ ]:
# Check if running on Google Colab
import os
IS_COLAB = 'google.colab' in str(get_ipython())
print(f"Running on Colab: {IS_COLAB}")

if IS_COLAB:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive')  # Change to your working directory

# Add repo to path
import sys
sys.path.insert(0, '/home/deli/ETH_projects/ml-in-fcs/fts-diffusion-ref')
os.chdir('/home/deli/ETH_projects/ml-in-fcs/fts-diffusion-ref')

In [ ]:
# Install dependencies
import subprocess
import sys

packages = ['torch', 'numpy', 'pandas', 'scikit-learn', 'matplotlib', 'yfinance', 'tqdm']

for package in packages:
    try:
        __import__(package)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

print("\n✓ All dependencies installed!")

In [ ]:
# Import all modules
import torch
import torch.optim as optim
torch.set_default_dtype(torch.float)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# FTS-Diffusion modules
from utils.load_data import get_fts, load_actual_fts
from models.model_params import prm_params, pgm_params, pem_params
from models.pattern_recognition_module import train_recognition_module
from models.train_ftsdiffusion import (
    train_ftsdiffusion,
    _has_recognition_artifacts,
    _has_generation_artifact,
    _has_evolution_artifact
)
from models.sampling import sampling_timeseries_ftsdiffusion
from models.load_models import load_ftsdiffusion

print("✓ All modules imported successfully!")

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Device setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Section 1: Data Loading

**Call chain:**
- `get_fts(ticker, fts_name, start_date, end_date)` → Downloads data from Yahoo Finance
- `load_actual_fts(dataname)` → Loads the preprocessed time series

In [ ]:
# Configuration
DATANAME = 'sp500'  # Dataset name
TICKER = '^GSPC'     # S&P 500
START_DATE = '1980-01-01'
END_DATE = '2020-01-01'

print(f"Downloading {TICKER} from {START_DATE} to {END_DATE}...")
get_fts(ticker=TICKER, fts_name=DATANAME, start_date=START_DATE, end_date=END_DATE)
print("✓ Data downloaded!")

In [ ]:
# Load the time series
fts = load_actual_fts(DATANAME).squeeze()

print(f"Time series shape: {fts.shape}")
print(f"Mean: {fts.mean():.4f}, Std: {fts.std():.4f}")
print(f"Min: {fts.min():.4f}, Max: {fts.max():.4f}")

# Plot the raw time series
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(fts, linewidth=0.8, alpha=0.8)
ax.set_xlabel('Days')
ax.set_ylabel('Log Returns')
ax.set_title('S&P 500 Log Returns (1980-2020)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✓ Data loaded: {len(fts)} observations")

## Section 2: Model Training

**Call chain:**
```
train_ftsdiffusion(fts)
  ├─ train_ftsdiffusion_recognition()
  │   └─ train_recognition_module() [SISC clustering]
  ├─ train_ftsdiffusion_generation()
  │   └─ train_generation_module() [Scaling AE + Diffusion]
  └─ train_ftsdiffusion_evolution()
      └─ train_evolution_module() [Pattern transitions]
```

**Note:** The code checks for existing artifacts and skips redundant training.

In [ ]:
# Check what artifacts already exist
print("Checking for existing training artifacts...")
has_recognition = _has_recognition_artifacts()
has_generation = _has_generation_artifact()
has_evolution = _has_evolution_artifact()

print(f"Recognition artifacts: {'✓' if has_recognition else '✗'}")
print(f"Generation artifacts: {'✓' if has_generation else '✗'}")
print(f"Evolution artifacts: {'✓' if has_evolution else '✗'}")

if has_recognition and has_generation and has_evolution:
    print("\n✓ All artifacts found. Skipping training.")
else:
    print("\n⚠ Some artifacts missing. Will train the model.")

In [ ]:
# Train the full FTS-Diffusion model
# This will skip stages that already have artifacts saved
print("Starting FTS-Diffusion training...\n")
print("="*60)

train_ftsdiffusion(fts, train_test_split=0.8, store_model=True)

print("="*60)
print("\n✓ Training complete!")
print("\nArtifacts saved:")
print("  - res/sisc_*.csv (patterns, labels, segments)")
print("  - trained_models/*.pth (generation & evolution models)")

## Section 3: Analyze Learned Patterns

The SISC module discovered recurring patterns in the time series.

In [ ]:
# Load SISC patterns
dataname = prm_params['dataname']
n_clusters = prm_params['k']
l_min, l_max = prm_params['l_min'], prm_params['l_max']
barycenter = prm_params['barycenter']
init_strategy = prm_params['init_strategy']

dict_init = {'kmeans++': 'kmpp', 'random_sample': 'rs', 'random_noise': 'rn'}
prefix = f"res/sisc_{dataname}_k{n_clusters}_l{l_min}-{l_max}_{barycenter[:4]}_{dict_init[init_strategy]}"

patterns = pd.read_csv(prefix + '_centroids.csv', header=None).values
labels = pd.read_csv(prefix + '_labels.csv', header=None).values.flatten()
subsequences = pd.read_csv(prefix + '_subsequences.csv', header=None).values

print(f"Discovered {n_clusters} patterns")
print(f"Pattern shape: {patterns.shape}")
print(f"Total segments: {len(labels)}")

In [ ]:
# Visualize the discovered patterns
fig, axes = plt.subplots(3, 5, figsize=(15, 8))
axes = axes.flatten()

for i in range(min(n_clusters, 15)):
    axes[i].plot(patterns[i], linewidth=2, color='steelblue')
    axes[i].set_title(f'Pattern {i}')
    axes[i].grid(alpha=0.3)
    axes[i].set_ylabel('Normalized Value')

# Hide unused subplots
for i in range(n_clusters, 15):
    axes[i].axis('off')

fig.suptitle('Discovered Recurring Patterns (SISC Centroids)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Analyze pattern frequency
unique, counts = np.unique(labels, return_counts=True)
pattern_freq = pd.DataFrame({'Pattern ID': unique, 'Frequency': counts})
pattern_freq = pattern_freq.sort_values('Frequency', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(pattern_freq['Pattern ID'], pattern_freq['Frequency'], color='steelblue')
ax.set_xlabel('Pattern ID')
ax.set_ylabel('Frequency')
ax.set_title('Pattern Frequency in Training Data')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(pattern_freq.to_string(index=False))

## Section 4: Generate Synthetic Time Series

**Call chain:**
```
sampling_timeseries_ftsdiffusion(T)
  ├─ load_ftsdiffusion() [Load trained models]
  └─ For each step:
      ├─ state_evolution_ftsdiffusion() [Predict next pattern]
      └─ segment_generation_ftsdiffusion() [Generate segment]
```

The loop generates segments autoregressively until reaching target length T.

In [ ]:
# Generate synthetic time series
TARGET_LENGTH = 2000  # Generate 2000 observations

print(f"Generating synthetic time series of length {TARGET_LENGTH}...\n")
synthetic_ts = sampling_timeseries_ftsdiffusion(
    T=TARGET_LENGTH,
    sample_idx=None,  # Random initialization
    plot_ts=False,
    store_ts=False,
    dataname=DATANAME
)

print(f"\n✓ Generated synthetic series shape: {synthetic_ts.shape}")
print(f"Mean: {synthetic_ts.mean():.4f}, Std: {synthetic_ts.std():.4f}")

In [ ]:
# Compare real vs synthetic time series
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Real time series (first 2000 points for comparison)
axes[0].plot(fts[:2000], linewidth=0.8, alpha=0.8, color='steelblue')
axes[0].set_ylabel('Log Returns')
axes[0].set_title('Real Time Series (S&P 500, 1980-2020)')
axes[0].grid(alpha=0.3)

# Synthetic time series
axes[1].plot(synthetic_ts, linewidth=0.8, alpha=0.8, color='coral')
axes[1].set_ylabel('Log Returns')
axes[1].set_xlabel('Days')
axes[1].set_title('Synthetic Time Series (Generated by FTS-Diffusion)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Section 5: Distribution Analysis

Compare stylized facts between real and synthetic data.

In [ ]:
# Helper functions for stylized facts
def compute_stylized_facts(returns):
    """Compute key stylized facts of financial time series"""
    facts = {}
    
    # Basic statistics
    facts['mean'] = np.mean(returns)
    facts['std'] = np.std(returns)
    facts['skewness'] = pd.Series(returns).skew()
    facts['kurtosis'] = pd.Series(returns).kurtosis()
    
    # Autocorrelation
    facts['acf_1'] = np.corrcoef(returns[:-1], returns[1:])[0, 1]
    
    # Volatility clustering (ACF of absolute returns)
    abs_returns = np.abs(returns)
    facts['acf_abs_1'] = np.corrcoef(abs_returns[:-1], abs_returns[1:])[0, 1]
    
    return facts

# Compute for both series
real_facts = compute_stylized_facts(fts[:2000])
synth_facts = compute_stylized_facts(synthetic_ts[:2000])

# Create comparison table
comparison = pd.DataFrame({
    'Real Data': real_facts,
    'Synthetic Data': synth_facts,
})
comparison['Difference'] = np.abs(comparison['Real Data'] - comparison['Synthetic Data'])

print("\nStylized Facts Comparison:")
print(comparison.round(4))

In [ ]:
# Visualize distribution comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Return distributions
axes[0, 0].hist(fts[:2000], bins=50, alpha=0.6, label='Real', color='steelblue')
axes[0, 0].hist(synthetic_ts[:2000], bins=50, alpha=0.6, label='Synthetic', color='coral')
axes[0, 0].set_xlabel('Log Returns')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Return Distributions')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Q-Q plots
from scipy import stats
stats.probplot(fts[:2000], dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Real Data Q-Q Plot')
axes[0, 1].grid(alpha=0.3)

stats.probplot(synthetic_ts[:2000], dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Synthetic Data Q-Q Plot')
axes[1, 1].grid(alpha=0.3)

# ACF comparison
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(pd.Series(fts[:2000]).diff().dropna(), ax=axes[1, 0], lags=40)
axes[1, 0].set_title('ACF of Real Data')
axes[1, 0].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Section 6: Multiple Synthetic Realizations

Generate multiple independent samples to assess variability.

In [ ]:
# Generate multiple synthetic realizations
N_SAMPLES = 5
samples = []

print(f"Generating {N_SAMPLES} independent synthetic realizations...\n")

for i in range(N_SAMPLES):
    print(f"Sample {i+1}/{N_SAMPLES}")
    sample = sampling_timeseries_ftsdiffusion(
        T=TARGET_LENGTH,
        sample_idx=None,
        plot_ts=False,
        store_ts=False,
        dataname=DATANAME
    )
    samples.append(sample)

samples = np.array(samples)
print(f"\n✓ Generated {samples.shape[0]} samples of shape {samples.shape[1]}")

In [ ]:
# Plot all realizations
fig, axes = plt.subplots(N_SAMPLES + 1, 1, figsize=(14, 3*(N_SAMPLES+1)))

# Real data
axes[0].plot(fts[:TARGET_LENGTH], linewidth=0.8, color='steelblue')
axes[0].set_ylabel('Returns')
axes[0].set_title('Real Time Series', fontweight='bold')
axes[0].grid(alpha=0.3)

# Synthetic samples
for i in range(N_SAMPLES):
    axes[i+1].plot(samples[i], linewidth=0.8, color='coral')
    axes[i+1].set_ylabel('Returns')
    axes[i+1].set_title(f'Synthetic Realization {i+1}')
    axes[i+1].grid(alpha=0.3)

axes[-1].set_xlabel('Days')
plt.tight_layout()
plt.show()

In [ ]:
# Compute stylized facts for all samples
print("\nStylized Facts Across All Samples:")
print("="*70)

stats_list = []

# Real data
real_stats = compute_stylized_facts(fts[:TARGET_LENGTH])
stats_list.append(('Real Data', real_stats))

# Synthetic samples
for i, sample in enumerate(samples):
    synth_stats = compute_stylized_facts(sample)
    stats_list.append((f'Synthetic {i+1}', synth_stats))

# Create summary table
summary_df = pd.DataFrame([v for k, v in stats_list], index=[k for k, v in stats_list])
print(summary_df.round(4))

# Compute mean and std across synthetic samples
synth_df = summary_df.iloc[1:]
print("\nSynthetic Data Mean (across 5 realizations):")
print(synth_df.mean().round(4))
print("\nSynthetic Data Std (across 5 realizations):")
print(synth_df.std().round(4))

## Section 7: Model Checkpoints & Paths

Summary of where artifacts are saved.

In [ ]:
import os
import glob

print("\nFTS-Diffusion Artifacts Location:")
print("="*70)

# SISC artifacts
print("\n[SISC Pattern Recognition]")
sisc_files = glob.glob(f"res/sisc_*")
for f in sisc_files:
    size = os.path.getsize(f) / 1024  # KB
    print(f"  {os.path.basename(f):40s} {size:8.2f} KB")

# Model checkpoints
print("\n[Trained Models]")
model_files = glob.glob(f"trained_models/*.pth")
for f in model_files:
    size = os.path.getsize(f) / (1024*1024)  # MB
    print(f"  {os.path.basename(f):40s} {size:8.2f} MB")

# Data files
print("\n[Data Files]")
data_files = glob.glob(f"data/*.npy") + glob.glob(f"data/*.csv")
for f in data_files[:10]:  # Show first 10
    if os.path.getsize(f) < 10*1024*1024:  # Skip large files
        size = os.path.getsize(f) / 1024  # KB
        print(f"  {os.path.basename(f):40s} {size:8.2f} KB")

## Summary

This notebook demonstrates the complete FTS-Diffusion pipeline:

### Training Pipeline:
1. **SISC Pattern Recognition**: Discovered K=14 recurring patterns
2. **Generation Module**: Scaling autoencoder + conditional diffusion model
3. **Evolution Module**: Markov transition network for pattern sequences

### Sampling Pipeline:
1. Initialize with random real segment
2. Predict next pattern via evolution module
3. Generate new segment via generation module
4. Continue autoregessively until reaching target length

### Key Results:
- Generated synthetic series preserve stylized facts (mean, variance, skewness, kurtosis)
- Volatility clustering captured (ACF of absolute returns)
- Multiple independent realizations available

### Next Steps:
- Evaluate on downstream tasks (TMTR, TATR predictability)
- Compute statistical tests (KS, AD, etc.)
- Analyze long-range dependence (Hurst exponent)